In [6]:
# ===============================================
# Script de test d'intégrité des données MongoDB
# ===============================================
# Objectif : Vérifier que les collections MongoDB contiennent des données correctes
# et bien typées avant de les utiliser pour l'analyse ou le chargement dans d'autres systèmes.
# ===============================================

from pymongo import MongoClient
import pandas as pd

# --------------------------
# Connexion à la base MongoDB
# --------------------------
# On se connecte à un serveur MongoDB local (localhost:27017)
client = MongoClient("mongodb://localhost:27017/")

# Nom de la base de données
db_name = "GreenAndCoop"
db = client[db_name]

# --------------------------------------
# Fonction pour charger une collection
# --------------------------------------
def load_collection_from_mongo(collection_name):
    """
    Charge une collection MongoDB dans un DataFrame pandas.
    
    Paramètre :
        collection_name (str) : Nom de la collection à charger
    
    Retour :
        pandas.DataFrame : Contient toutes les données de la collection
    """
    # On ignore le champ _id en MongoDB car il n'est pas nécessaire pour l'analyse
    data = list(db[collection_name].find({}, {"_id": 0}))
    return pd.DataFrame(data)

# --------------------------------------
# Fonction pour exécuter les tests d'intégrité
# --------------------------------------
def run_quality_checks(df, collection_name):
    """
    Tests d'intégrité des données avec affichage détaillé pour chaque colonne.
    """
    print(f"\n Tests pour la collection : {collection_name}")

    # Test 1 : collection non vide
    assert not df.empty, f"{collection_name} est vide"

    # Test 2 : valeurs manquantes par colonne
    print("\nValeurs manquantes par colonne :")
    print(df.isnull().sum())

    # Test 3 : doublons
    try:
        duplicates = df.duplicated().sum()
        print(f"\nDoublons détectés : {duplicates}")
    except TypeError:
        print("Test des doublons ignoré (colonnes contenant des listes ou objets complexes)")

    # Test 4 : types des colonnes
    print("\nTypes des colonnes :")
    print(df.dtypes)

    # Test 5 : aperçu des premières lignes
    print("\nAperçu des données :")
    print(df.head())

    print(f"\n Tests OK pour {collection_name}")


# --------------------------------------
# Liste des collections à tester
# --------------------------------------
collections_to_test = [
    "InfoClimat",
    "WeatherIchtegem",
    "WeatherLaMadeleine"
]

# --------------------------------------
# Exécution des tests pour chaque collection
# --------------------------------------
for collection_name in collections_to_test:
    df_collection = load_collection_from_mongo(collection_name)
    run_quality_checks_on_dataframe(df_collection, collection_name)

print("\n Tous les tests d’intégrité ont été exécutés avec succès !")



 Tests d'intégrité pour la collection : InfoClimat
Valeurs manquantes dans 'InfoClimat' : 0
 Test des doublons ignoré (colonnes avec des objets complexes)

Types des colonnes :
status      object
errors      object
data        object
stations    object
metadata    object
hourly      object
dtype: object
 Tous les tests sont OK pour la collection 'InfoClimat'

 Tests d'intégrité pour la collection : WeatherIchtegem
Valeurs manquantes dans 'WeatherIchtegem' : 14
Doublons détectés dans 'WeatherIchtegem' : 1899

Types des colonnes :
Wind              object
Humidity          object
Temperature      float64
Pressure         float64
Speed            float64
Dew Point         object
Precip. Rate.     object
Time              object
UV                 int64
Solar             object
Gust              object
Precip_Accum     float64
dtype: object
 Tous les tests sont OK pour la collection 'WeatherIchtegem'

 Tests d'intégrité pour la collection : WeatherLaMadeleine
Valeurs manquantes dans 'Weat